# Dunnhumby M5_A·M5_B — 마지막 주 seed50

**현재 확정 명칭:** M5_A = 적응형 M2 + M3 / M5_B = 적응형 M2 + M4(고객 내 평균1).
과거 M4 변형을 구분했던 M5-A/B와는 다른 현재 조합입니다.

- **새 학습은 두 번만:** M5_A → M5_B, 한 L4 GPU에서 순차 실행.
- 기존 **M1·M2·M3·M4·M5_C의 seed50 결과값만 재사용**합니다. 참조모형 학습·재평가 없음.
- 각 결합모형은 새 초기값에서 하나의 optimizer로 함께 학습합니다. 기존 가중치를 동결하거나 이어 붙이지 않습니다.
- train DAY 1~704 / test DAY 705~711, 300 epoch 마지막 점만 1회 평가. CLV 관측창은 기존과 같은 학습 말미 365일.
- ID64, LightGCN2층, batch8192, lr0.0005, ID L2 0.001, uniform 음성1개, 전체 카탈로그, 학습 구매쌍 제외.
- 단일시드·이미 관찰한 test의 추가 비교입니다. 유의성·새 확증·CLV 귀속을 주장하지 않고, 두 모형의 불리한 결과도 함께 보고합니다.

런타임 → 런타임 유형 변경 → **L4 GPU** 선택 후 위에서부터 실행하세요.
중단되면 같은 노트북·경로로 다시 실행합니다. 완료 arm은 건너뛰고 미완료 arm은 저장 epoch에서 재개합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU는 {GPU}입니다. 요청한 L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '828de460d197e1385e61b8c9efb973aaae857c95'
REPO = Path('/content/clv-dunnhumby-m5-followup-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 기존 결과 확인·공통 입력 준비 (학습 없음)
Drive의 기존 M2·M4 최종10시드 결과 폴더에서 원본을 자동 확인합니다. M4 결과에는 M1·M3·M5_C 참조가 함께 있습니다.
원본과 일치하는 파일이 없으면 **학습 전에 중단**하며, 대체 모형을 학습하지 않습니다.

폴더를 옮겼다면 아래 `m2_reference`·`m4_reference`만 지정하세요. 원본 ZIP 경로도 허용합니다:
`clv_m2_adaptive_dunnhumby_final10_results.zip`, `clv_m4_user_centered_final10_results.zip`.
실행을 재개할 때는 처음 사용한 참조·저장 경로를 유지하세요.

In [ ]:
import clv_m5_ab_adaptive_lastweek_seed49 as screen
cfg = screen.configure(dataset='dunnhumby', seed=50,
    # m2_reference='/content/drive/MyDrive/.../clv_m2_adaptive_dunnhumby_final10_results.zip',
    # m4_reference='/content/drive/MyDrive/.../clv_m4_user_centered_final10_results.zip',
)
print('M2 참조:', cfg.m2_reference)
print('M1/M3/M4/M5_C 참조:', cfg.m4_reference)
print('새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('재사용:', [screen.labels_for(cfg)[m] for m in prepared['references']['absolute'].model_id])
print('새 학습 순서: M5_A(M2+M3) → M5_B(M2+M4), seed50 각300epoch')

## 2. 두 결합모형 순차 학습
학습 중에는 loss만 표시하며 test는 확인하지 않습니다. A가 끝나면 결과를 Drive에 저장하고 B를 시작합니다.
학습 중 노트북의 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 3. 같은 seed50의 일곱 모형 비교
참조는 10시드 평균이 아니라 **50번 시드 한 행**입니다. 이 표는 핵심지표만 표시하며 전체142지표·세그먼트·노출은 CSV에 저장합니다.
학습이 중단된 경우에도 준비 셀 실행 후 이 셀로 완료된 arm의 저장 결과만 볼 수 있습니다. 추가 평가·학습은 없습니다.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
           'price_purchase_amount_weighted_hit@10', 'vndcg@10',
           'price_purchase_amount_weighted_hit@20', 'vndcg@20',
           'price_purchase_amount_weighted_hit@50', 'vndcg@50']
display(result['absolute'].set_index('model')[metrics].T.reindex(
    columns=['M1', 'M2', 'M3', 'M4', 'M5_A', 'M5_B', 'M5_C']))
display(result['comparison'].query("reference == 'M1'").pivot(
    index='metric', columns='model', values='relative_change_pct').reindex(metrics)
    if len(result['comparison']) else result['comparison'])
print('M1 대비 상대변화(%), 1시드 결과 — 유의성 주장 없음')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

In [ ]:
from google.colab import files
output = screen.export_zip(prepared, '/content/dh_m5_ab_adaptive_lastweek_seed50_results.zip')
files.download(output)